# Mini-Projet : Spark avec Docker

**Binôme :** Mazilda ZEHRAOUI et Manjakasoa RANJATOSON

Analyse des ventes d'une chaîne de magasins (`purchases.txt`) avec Apache Spark : RDD, DataFrame API, Spark SQL et un mini-pipeline ETL vers Parquet.

| Étape | Contenu |
|---|---|
| 1 | Environnement Docker et SparkSession |
| 2 | Chargement et exploration |
| 3 | Traitement avec les RDD |
| 4 | DataFrame API |
| 5 | Spark SQL |
| 6 | Pipeline ETL et Parquet |

# Étape 1 : Préparation de l'environnement Docker

Le conteneur est lancé avec `docker compose up -d`, puis Jupyter est ouvert sur `http://localhost:8888`. On initialise ensuite la session Spark.

In [34]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("MiniProjet_Spark").master("local[2]").getOrCreate()
print("Master =", spark.sparkContext.master)

Master = local[2]


### 1. Quel est le rôle de Docker dans ce mini-projet ?
Docker nous permet de lancer Spark et Jupyter dans un conteneur, sans avoir à installer toutes les dépendances sur nos ordinateurs.

### 2. Pourquoi utilisons-nous une image Docker contenant PySpark ?
L'image contient déjà Python, PySpark et Jupyter. On peut donc directement travailler avec Spark sans devoir tout configurer nous-mêmes.

### 3. Quelle est la différence entre un conteneur Docker et une machine virtuelle ?
Un conteneur Docker partage le noyau du système sur lequel il tourne, alors qu'une machine virtuelle possède son propre système d'exploitation et son propre noyau. Docker est donc généralement plus léger et plus rapide à démarrer.

### 4. À quoi sert le fichier docker-compose.yml ?
Il sert à configurer et lancer notre conteneur. Dans notre cas, il précise l'image PySpark à utiliser, le port pour accéder à Jupyter et le dossier partagé entre notre ordinateur et le conteneur.

### 5. Que signifie le mode d'exécution local[2] de Spark ?
Cela signifie que Spark tourne sur une seule machine avec 2 threads, ce qui lui permet d'exécuter jusqu'à 2 tâches en parallèle.

### 6. Quel est le rôle de SparkSession ?
C'est le point d'entrée de Spark. On l'utilise pour travailler avec les DataFrames, charger des données et lancer des requêtes SQL.


# Étape 2 : Chargement et exploration des données

Le fichier n'a pas d'en-tête et ses champs sont séparés par des tabulations. 

In [35]:
import os
from pyspark.sql.functions import col
print(os.listdir("/home/jovyan/work"))

['output', 'MiniProject_Spark.ipynb', 'purchases.txt', '.ipynb_checkpoints', 'notebooks']


In [36]:
with open("/home/jovyan/work/purchases.txt", "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline().rstrip())

2012-01-01	09:00	San Jose	Men's Clothing	214.05	Amex
2012-01-01	09:00	Fort Worth	Women's Clothing	153.57	Visa
2012-01-01	09:00	San Diego	Music	66.08	Cash
2012-01-01	09:00	Pittsburgh	Pet Supplies	493.51	Discover
2012-01-01	09:00	Omaha	Children's Clothing	235.63	MasterCard


In [37]:
# 1. Charger les données dans un DataFrame
df = spark.read.csv(
    "/home/jovyan/work/purchases.txt",
    sep="\t",
    header=False
)

In [38]:
# 2. Définir les noms des colonnes
print(df.columns)
df = df.toDF("Date", "Heure", "Magasin", "Produit", "Cout", "Paiement")
print(df.columns)

['_c0', '_c1', '_c2', '_c3', '_c4', '_c5']
['Date', 'Heure', 'Magasin', 'Produit', 'Cout', 'Paiement']


In [39]:
# 3. Afficher les 10 premières lignes
df.show(10)

+----------+-----+--------------+--------------------+------+----------+
|      Date|Heure|       Magasin|             Produit|  Cout|  Paiement|
+----------+-----+--------------+--------------------+------+----------+
|2012-01-01|09:00|      San Jose|      Men's Clothing|214.05|      Amex|
|2012-01-01|09:00|    Fort Worth|    Women's Clothing|153.57|      Visa|
|2012-01-01|09:00|     San Diego|               Music| 66.08|      Cash|
|2012-01-01|09:00|    Pittsburgh|        Pet Supplies|493.51|  Discover|
|2012-01-01|09:00|         Omaha| Children's Clothing|235.63|MasterCard|
|2012-01-01|09:00|      Stockton|      Men's Clothing|247.18|MasterCard|
|2012-01-01|09:00|        Austin|             Cameras| 379.6|      Visa|
|2012-01-01|09:00|      New York|Consumer Electronics| 296.8|      Cash|
|2012-01-01|09:00|Corpus Christi|                Toys| 25.38|  Discover|
|2012-01-01|09:00|    Fort Worth|                Toys|213.88|      Visa|
+----------+-----+--------------+------------------

In [40]:
# 4. Afficher le schéma du DataFrame
df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Heure: string (nullable = true)
 |-- Magasin: string (nullable = true)
 |-- Produit: string (nullable = true)
 |-- Cout: string (nullable = true)
 |-- Paiement: string (nullable = true)



In [41]:
# 5. Vérifier et convertir Cout en numerique
print(f"Cout type : {df.schema["Cout"].dataType}")

df = df.withColumn("Cout", col("Cout").cast("double"))

df.printSchema()


Cout type : StringType()
root
 |-- Date: string (nullable = true)
 |-- Heure: string (nullable = true)
 |-- Magasin: string (nullable = true)
 |-- Produit: string (nullable = true)
 |-- Cout: double (nullable = true)
 |-- Paiement: string (nullable = true)



In [42]:
# 6. Nombre total de transactions
nb_transactions = df.count()

print("Nombre total de transactions :", nb_transactions)

Nombre total de transactions : 4138476


**Interprétation :** à la lecture, toutes les colonnes sont de type `string`. On convertit `Cout` en `double` pour effectuer des comparaisons et des calculs numériques (somme, moyenne). Le fichier contient au total **4 138 476 transactions**, qui serviront de base aux analyses suivantes.

# Étape 3 : Traitement avec les RDD

On repart du fichier brut `purchases.txt` avec l'API bas niveau de Spark (RDD). Chaque ligne contient 6 champs séparés par une tabulation : Date, Heure, Magasin, Produit, Coût, Paiement.

In [43]:
sc = spark.sparkContext

# Création du RDD à partir du fichier texte
rdd = sc.textFile("/home/jovyan/work/purchases.txt")

# 1. Afficher les cinq premiers éléments
for ligne in rdd.take(5):
    print(ligne)

2012-01-01	09:00	San Jose	Men's Clothing	214.05	Amex
2012-01-01	09:00	Fort Worth	Women's Clothing	153.57	Visa
2012-01-01	09:00	San Diego	Music	66.08	Cash
2012-01-01	09:00	Pittsburgh	Pet Supplies	493.51	Discover
2012-01-01	09:00	Omaha	Children's Clothing	235.63	MasterCard


Chaque élément du RDD est pour l'instant une simple chaîne de caractères (une ligne du fichier).

In [44]:
# 2. map() : découper chaque ligne en liste de champs
rdd_champs = rdd.map(lambda ligne: ligne.split("\t"))

# On écarte les éventuelles lignes mal formées (≠ 6 champs)
rdd_champs = rdd_champs.filter(lambda champs: len(champs) == 6)

rdd_champs.take(5)

[['2012-01-01', '09:00', 'San Jose', "Men's Clothing", '214.05', 'Amex'],
 ['2012-01-01', '09:00', 'Fort Worth', "Women's Clothing", '153.57', 'Visa'],
 ['2012-01-01', '09:00', 'San Diego', 'Music', '66.08', 'Cash'],
 ['2012-01-01', '09:00', 'Pittsburgh', 'Pet Supplies', '493.51', 'Discover'],
 ['2012-01-01',
  '09:00',
  'Omaha',
  "Children's Clothing",
  '235.63',
  'MasterCard']]

In [45]:
# 3. Extraire les couples (Magasin, Cout), avec Cout converti en nombre
rdd_magasin_cout = rdd_champs.map(lambda champs: (champs[2], float(champs[4])))

rdd_magasin_cout.take(5)

[('San Jose', 214.05),
 ('Fort Worth', 153.57),
 ('San Diego', 66.08),
 ('Pittsburgh', 493.51),
 ('Omaha', 235.63)]

In [46]:
# 4. filter() : garder les transactions dont le montant est > 100
rdd_sup_100 = rdd_magasin_cout.filter(lambda x: x[1] > 100)

print("Transactions > 100 :", rdd_sup_100.count())
rdd_sup_100.take(5)

Transactions > 100 : 3310957


[('San Jose', 214.05),
 ('Fort Worth', 153.57),
 ('Pittsburgh', 493.51),
 ('Omaha', 235.63),
 ('Stockton', 247.18)]

In [47]:
# 5. reduceByKey() : montant total des ventes par magasin
rdd_ca_magasin = rdd_magasin_cout.reduceByKey(lambda a, b: a + b)

print("Nombre de magasins :", rdd_ca_magasin.count())
# Affichage des 10 plus gros CA (tri décroissant)
for magasin, total in rdd_ca_magasin.sortBy(lambda x: x[1], ascending=False).take(10):
    print(f"{magasin:<20} {total:,.2f}")

Nombre de magasins : 103
Philadelphia         10,190,080.26
Durham               10,153,890.21
Laredo               10,144,604.98
Newark               10,144,052.80
Cincinnati           10,139,505.74
Washington           10,139,363.39
Irving               10,133,944.08
Fort Wayne           10,132,594.02
Baton Rouge          10,131,273.23
Sacramento           10,123,468.18


In [48]:
# Variante : total par magasin uniquement sur les ventes > 100
rdd_ca_sup_100 = rdd_sup_100.reduceByKey(lambda a, b: a + b)

rdd_ca_sup_100.sortBy(lambda x: x[1], ascending=False).take(5)

[('Philadelphia', 9785891.520000005),
 ('Durham', 9761328.09999999),
 ('Laredo', 9740913.840000002),
 ('Newark', 9737845.659999985),
 ('Washington', 9736221.449999996)]

**Interprétation :** on obtient un chiffre d'affaires par magasin. Ces totaux sont les mêmes que ceux calculés avec la DataFrame API et Spark SQL (étapes 4 et 5), mais le code RDD est plus long : il faut découper les lignes et convertir les types à la main.

### Questions

**1. Différence entre `map()` et `filter()` ?**  
`map()` applique une fonction à chaque élément et renvoie un élément transformé pour chacun (même nombre d'éléments). `filter()` ne modifie pas les éléments : il garde seulement ceux qui vérifient une condition (moins d'éléments ou autant).

**2. Rôle de `reduceByKey()` ?**  
Sur un RDD de couples (clé, valeur), il regroupe les valeurs qui ont la même clé et les combine avec une fonction (ici l'addition). Il fait d'abord une réduction locale dans chaque partition, puis combine les résultats, ce qui limite les données échangées entre les partitions (shuffle).

**3. Différence entre une transformation et une action ?**  
Une transformation (`map`, `filter`, `reduceByKey`…) crée un nouveau RDD et est *paresseuse* (lazy) : Spark l'enregistre sans l'exécuter. Une action (`take`, `count`, `collect`…) déclenche réellement le calcul et renvoie un résultat au driver.

# Étape 4 : Analyse avec la DataFrame API

In [49]:
from pyspark.sql.functions import sum, avg

# 1. Afficher uniquement les colonnes Magasin, Produit et Cout
df.select("Magasin", "Produit", "Cout").show(10)

+--------------+--------------------+------+
|       Magasin|             Produit|  Cout|
+--------------+--------------------+------+
|      San Jose|      Men's Clothing|214.05|
|    Fort Worth|    Women's Clothing|153.57|
|     San Diego|               Music| 66.08|
|    Pittsburgh|        Pet Supplies|493.51|
|         Omaha| Children's Clothing|235.63|
|      Stockton|      Men's Clothing|247.18|
|        Austin|             Cameras| 379.6|
|      New York|Consumer Electronics| 296.8|
|Corpus Christi|                Toys| 25.38|
|    Fort Worth|                Toys|213.88|
+--------------+--------------------+------+
only showing top 10 rows


In [50]:
# 2. Filtrer les ventes dont le montant est supérieur à 100
df.filter(col("Cout") > 100).show(10)

+----------+-----+----------+--------------------+------+----------+
|      Date|Heure|   Magasin|             Produit|  Cout|  Paiement|
+----------+-----+----------+--------------------+------+----------+
|2012-01-01|09:00|  San Jose|      Men's Clothing|214.05|      Amex|
|2012-01-01|09:00|Fort Worth|    Women's Clothing|153.57|      Visa|
|2012-01-01|09:00|Pittsburgh|        Pet Supplies|493.51|  Discover|
|2012-01-01|09:00|     Omaha| Children's Clothing|235.63|MasterCard|
|2012-01-01|09:00|  Stockton|      Men's Clothing|247.18|MasterCard|
|2012-01-01|09:00|    Austin|             Cameras| 379.6|      Visa|
|2012-01-01|09:00|  New York|Consumer Electronics| 296.8|      Cash|
|2012-01-01|09:00|Fort Worth|                Toys|213.88|      Visa|
|2012-01-01|09:00|    Austin|             Cameras|469.63|MasterCard|
|2012-01-01|09:00|Greensboro|                DVDs|290.82|MasterCard|
+----------+-----+----------+--------------------+------+----------+
only showing top 10 rows


In [51]:
# 3. Trier ces ventes par montant décroissant
df.filter(col("Cout") > 100).orderBy(col("Cout").desc()).show(10)

+----------+-----+-----------+--------------------+------+----------+
|      Date|Heure|    Magasin|             Produit|  Cout|  Paiement|
+----------+-----+-----------+--------------------+------+----------+
|2012-08-27|17:02|Los Angeles|Consumer Electronics|499.99|  Discover|
|2012-10-07|15:19|    Phoenix| Children's Clothing|499.99|      Amex|
|2012-09-08|12:33|    Detroit|              Crafts|499.99|MasterCard|
|2012-08-13|16:47|      Plano|         Video Games|499.99|      Amex|
|2012-09-12|12:28|      Omaha|           Computers|499.99|  Discover|
|2012-07-12|17:54|     Boston|              Crafts|499.99|      Cash|
|2012-09-17|09:08|  St. Louis|               Books|499.99|      Cash|
|2012-07-08|09:49|     Dallas|                 CDs|499.99|MasterCard|
|2012-09-18|09:45| Pittsburgh|              Garden|499.99|      Amex|
|2012-08-10|14:48|   Honolulu|   Health and Beauty|499.99|  Discover|
+----------+-----+-----------+--------------------+------+----------+
only showing top 10 

In [52]:
# 4. Calculer le chiffre d'affaires total par magasin
df_ca_magasin = df.groupBy("Magasin").agg(sum("Cout").alias("CA_total"))

df_ca_magasin.show()

+---------------+--------------------+
|        Magasin|            CA_total|
+---------------+--------------------+
|North Las Vegas|1.0029652509999933E7|
|        Phoenix|1.0079076700000003E7|
|          Omaha| 1.002664233999998E7|
|      Anchorage|   9933500.399999995|
|        Anaheim|1.0076416359999968E7|
|     Greensboro|1.0033781390000068E7|
|         Dallas|1.0066548450000022E7|
|        Oakland|   9947292.520000033|
|         Laredo|1.0144604979999993E7|
|     Scottsdale|1.0037929849999957E7|
|    San Antonio|1.0014441700000005E7|
|    Bakersfield|1.0031208919999965E7|
|        Raleigh|1.0061442539999988E7|
|    Chula Vista|   9974951.340000018|
|   Philadelphia|1.0190080259999951E7|
|     Louisville|1.0008566470000014E7|
|    Los Angeles|1.0084576799999982E7|
|       Chandler|   9919559.859999968|
|     Sacramento|1.0123468179999966E7|
|   Indianapolis|1.0090272770000013E7|
+---------------+--------------------+
only showing top 20 rows


In [53]:
# 5. Identifier le magasin ayant le chiffre d'affaires le plus élevé
df_ca_magasin.orderBy(col("CA_total").desc()).show(1)

+------------+--------------------+
|     Magasin|            CA_total|
+------------+--------------------+
|Philadelphia|1.0190080259999951E7|
+------------+--------------------+
only showing top 1 row


In [54]:
# 6. Calculer le montant moyen des transactions par catégorie de produits
df.groupBy("Produit").agg(avg("Cout").alias("Montant_moyen")).show()

+--------------------+------------------+
|             Produit|     Montant_moyen|
+--------------------+------------------+
| Children's Clothing|250.03285014470467|
|      Sporting Goods|250.50487052693825|
|                 CDs|249.56965140693512|
|           Computers|250.22114966013262|
|Consumer Electronics|250.05276844198997|
|   Health and Beauty| 250.2823198805272|
|        Pet Supplies|249.52775143746985|
|                DVDs| 250.3505047899475|
|                Baby| 249.6463567715953|
|              Crafts|249.91688538361313|
|    Women's Clothing| 249.6607214518553|
|         Video Games| 249.7998392091628|
|               Books| 250.0174418483193|
|               Music|249.81746556593654|
|      Men's Clothing|250.05979707503488|
|             Cameras|249.86502110588287|
|              Garden|250.09380983426973|
|                Toys|249.88031652780342|
+--------------------+------------------+



# Étape 5 : Analyse avec Spark SQL

In [55]:
# 1. Créer une vue temporaire nommée ventes
df.createOrReplaceTempView("ventes")

In [56]:
# 2. Afficher les ventes dont le montant dépasse 100
spark.sql("""
    SELECT *
    FROM ventes
    WHERE Cout > 100
""").show(10)

+----------+-----+----------+--------------------+------+----------+
|      Date|Heure|   Magasin|             Produit|  Cout|  Paiement|
+----------+-----+----------+--------------------+------+----------+
|2012-01-01|09:00|  San Jose|      Men's Clothing|214.05|      Amex|
|2012-01-01|09:00|Fort Worth|    Women's Clothing|153.57|      Visa|
|2012-01-01|09:00|Pittsburgh|        Pet Supplies|493.51|  Discover|
|2012-01-01|09:00|     Omaha| Children's Clothing|235.63|MasterCard|
|2012-01-01|09:00|  Stockton|      Men's Clothing|247.18|MasterCard|
|2012-01-01|09:00|    Austin|             Cameras| 379.6|      Visa|
|2012-01-01|09:00|  New York|Consumer Electronics| 296.8|      Cash|
|2012-01-01|09:00|Fort Worth|                Toys|213.88|      Visa|
|2012-01-01|09:00|    Austin|             Cameras|469.63|MasterCard|
|2012-01-01|09:00|Greensboro|                DVDs|290.82|MasterCard|
+----------+-----+----------+--------------------+------+----------+
only showing top 10 rows


In [57]:
# 3. Calculer le chiffre d'affaires total par magasin
df_ca_sql = spark.sql("""
    SELECT
        Magasin,
        SUM(Cout) AS CA_total
    FROM ventes
    GROUP BY Magasin
""")

df_ca_sql.show()

+---------------+--------------------+
|        Magasin|            CA_total|
+---------------+--------------------+
|North Las Vegas|1.0029652509999933E7|
|        Phoenix|1.0079076700000003E7|
|          Omaha| 1.002664233999998E7|
|      Anchorage|   9933500.399999995|
|        Anaheim|1.0076416359999968E7|
|     Greensboro|1.0033781390000068E7|
|         Dallas|1.0066548450000022E7|
|        Oakland|   9947292.520000033|
|         Laredo|1.0144604979999993E7|
|     Scottsdale|1.0037929849999957E7|
|    San Antonio|1.0014441700000005E7|
|    Bakersfield|1.0031208919999965E7|
|        Raleigh|1.0061442539999988E7|
|    Chula Vista|   9974951.340000018|
|   Philadelphia|1.0190080259999951E7|
|     Louisville|1.0008566470000014E7|
|    Los Angeles|1.0084576799999982E7|
|       Chandler|   9919559.859999968|
|     Sacramento|1.0123468179999966E7|
|   Indianapolis|1.0090272770000013E7|
+---------------+--------------------+
only showing top 20 rows


In [58]:
# 4. Afficher les trois magasins ayant le chiffre d'affaires le plus élevé
spark.sql("""
    SELECT
        Magasin,
        SUM(Cout) AS CA_total
    FROM ventes
    GROUP BY Magasin
    ORDER BY CA_total DESC
    LIMIT 3
""").show()

+------------+--------------------+
|     Magasin|            CA_total|
+------------+--------------------+
|Philadelphia|1.0190080259999951E7|
|      Durham|1.0153890210000012E7|
|      Laredo|1.0144604979999993E7|
+------------+--------------------+



### 5. Comparaison entre Spark SQL et DataFrame API

Les deux méthodes donnent les mêmes résultats. Avec la DataFrame API, on utilise des fonctions Python comme `filter()` ou `groupBy()`, alors qu'avec Spark SQL, on écrit directement des requêtes SQL.

Par exemple, pour afficher les ventes supérieures à 100 :

**DataFrame API :**
```python
df.filter(col("Cout") > 100).show()
```

**Spark SQL :**
```sql
SELECT * FROM ventes WHERE Cout > 100
```

La principale différence est donc la syntaxe, puisque les deux approches utilisent le même moteur Spark et bénéficient de l'optimiseur Catalyst.


# Étape 6 : Mini-pipeline ETL avec PySpark

### Sauvegarde des résultats

In [62]:
# 1. Récupérer la DataFrame du chiffre d'affaires par magasin (étape 4)
df_resultat = df_ca_magasin

df_resultat.orderBy("Magasin").show(10)

+-----------+--------------------+
|    Magasin|            CA_total|
+-----------+--------------------+
|Albuquerque|1.0052311420000013E7|
|    Anaheim|1.0076416359999968E7|
|  Anchorage|   9933500.399999995|
|  Arlington|1.0072207970000042E7|
|    Atlanta|   9997146.699999977|
|     Aurora|   9992970.920000017|
|     Austin|1.0057158900000006E7|
|Bakersfield|1.0031208919999965E7|
|  Baltimore|1.0096521449999953E7|
|Baton Rouge| 1.013127323000006E7|
+-----------+--------------------+
only showing top 10 rows


In [63]:
# 2. Enregistrer au format Parquet (mode overwrite pour pouvoir réexécuter)
output_path = "/home/jovyan/work/output/ventes_par_magasin"

df_resultat.write.mode("overwrite").parquet(output_path)

In [64]:
# 3. Vérifier que les fichiers ont bien été créés
import os
print(os.listdir(output_path))

['part-00000-b94a50f8-0740-4f69-aab5-1e4414a55ab7-c000.snappy.parquet', '._SUCCESS.crc', '.part-00000-b94a50f8-0740-4f69-aab5-1e4414a55ab7-c000.snappy.parquet.crc', '_SUCCESS']


Le dossier contient un ou plusieurs fichiers `part-*.parquet` (les données) et un fichier `_SUCCESS` qui indique que l'écriture s'est bien terminée.

### Vérification des données enregistrées

In [66]:
# 1. Relire les données Parquet dans une nouvelle DataFrame
df_verification = spark.read.parquet(output_path)

# 2. Afficher les premières lignes
df_verification.orderBy("Magasin").show(10)

+-----------+--------------------+
|    Magasin|            CA_total|
+-----------+--------------------+
|Albuquerque|1.0052311420000013E7|
|    Anaheim|1.0076416359999968E7|
|  Anchorage|   9933500.399999995|
|  Arlington|1.0072207970000042E7|
|    Atlanta|   9997146.699999977|
|     Aurora|   9992970.920000017|
|     Austin|1.0057158900000006E7|
|Bakersfield|1.0031208919999965E7|
|  Baltimore|1.0096521449999953E7|
|Baton Rouge| 1.013127323000006E7|
+-----------+--------------------+
only showing top 10 rows


In [67]:
# 3. Vérifier le schéma
df_verification.printSchema()

root
 |-- Magasin: string (nullable = true)
 |-- CA_total: double (nullable = true)



In [68]:
# 4. Comparer avec les résultats d'avant l'enregistrement
print("Lignes avant :", df_resultat.count())
print("Lignes après :", df_verification.count())
print("Même schéma  :", df_resultat.schema == df_verification.schema)

# Différences dans les deux sens (0 = données identiques)
diff_1 = df_resultat.exceptAll(df_verification).count()
diff_2 = df_verification.exceptAll(df_resultat).count()
print("Lignes différentes :", diff_1 + diff_2)

Lignes avant : 103
Lignes après : 103
Même schéma  : True
Lignes différentes : 0


**Interprétation :** même nombre de lignes, même schéma (Magasin : string, CA_total : double) et aucune ligne différente. Le format Parquet a conservé les données et leurs types. L'ordre d'affichage peut varier, c'est pourquoi on trie par magasin avant de comparer visuellement.

### Questions

**1. Phases d'un pipeline ETL, et où on les trouve dans le projet :**
- **Extract** (extraire) : lecture de `purchases.txt` (étape 2 en DataFrame, étape 3 en RDD).
- **Transform** (transformer) : nommage des colonnes, conversion de `Cout` en double, filtres, agrégations par magasin et par catégorie de produit (étapes 3, 4 et 5).
- **Load** (charger) : écriture du CA par magasin au format Parquet (étape 6).

**2. Pourquoi utiliser Parquet en Big Data ?**  
C'est un format colonnaire, compressé, qui garde le schéma. Spark ne lit que les colonnes utiles, les fichiers sont plus petits et les lectures plus rapides. Il se découpe aussi bien pour un traitement en parallèle.

**3. Différence entre CSV et Parquet ?**  
CSV : texte lisible, stocké ligne par ligne, sans types (tout est lu comme du texte, il faut ré-inférer ou caster). Parquet : binaire, stocké par colonnes, avec compression intégrée, avec le schéma et les types conservés.

**4. Intérêt du mode `overwrite` ?**  
Il remplace le contenu du dossier s'il existe déjà. On peut donc relancer le notebook sans erreur (le mode par défaut échoue si le dossier existe) et sans accumuler d'anciens résultats.

**5. Différence entre `show()` et l'enregistrement en Parquet ?**  
`show()` affiche seulement quelques lignes à l'écran, et rien n'est conservé après l'arrêt de la session. Enregistrer en Parquet écrit toutes les données sur disque : elles sont persistantes et réutilisables par d'autres traitements.

# Conclusion

Les trois approches (RDD, DataFrame API, Spark SQL) donnent le même chiffre d'affaires par magasin. Les RDD demandent plus de code (découpage et conversion à la main). La DataFrame API et Spark SQL sont plus courts et plus lisibles, et ils passent par l'optimiseur de Spark (Catalyst). Le résultat final est enregistré en Parquet, puis relu à l'identique.